# Figure 2 — design diagnostics

**Whether they were comparable.** Treatment is nearly determined by centre in this
cohort, so the credibility of the primary estimate rests on two diagnostics rather than on the
estimate's own interval: what the weights achieved on the confounders, and where the two arms actually
overlap. [§9] requires both, and requires any covariate still imbalanced after weighting to be named
with its magnitude beside the estimate — so this is a main figure, not a supplementary one.

This notebook **composes**; it computes nothing. Every number comes off the cached run that
`manuscript.load()` returns, every sentence beneath the exhibit comes off `report.py`, and the
artefacts land in `out/manuscript/`. Section references `[§n]` point to `../statistical_analysis_plan.md`.

## 1. The run

One cached build of Stages 1-11, shared by all seven exhibits. The record below says what it depends on.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Independent of where the notebook was started: find the directory the shipped modules live in.
_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.py").exists())
sys.path[:0] = [str(_root), str(_root / "figures_and_tables")]

import config as C
import report
import manuscript as M

M.use_house_style()
ARM = C.TREATMENT_LABELS          # {0: "EVT alone", 1: "bridging"}
ARMS = sorted(ARM)                # control first

bundle = M.load()                 # the cached Stages 1-11 run; built once, then read in seconds
bundle.provenance

## 2. Panel A — what the weights achieved

One row per covariate, judged against the **full** `[§6]` confounder set rather than only the
covariates the propensity model used `[§9]`. The threshold is |SMD| < 0.10.

In [ ]:
# The Love plot shows the covariates weighting was expected to act on: those in a propensity model —
# the [§7] primary's or [§13]'s — and the four held out as negative controls. `penumbra_ml` is in
# neither, being core and Tmax>6 s restated, so no weighting was ever going to move it; it stays in
# the supplement's T05, where the full [§9] set is reported.
covariates = [r for r in bundle.balance.covariates
              if r.covariate.split(" = ")[0] in C.PS_COVARIATES_FULL]
smd = pd.DataFrame([(M.label(r.covariate), r.role, r.unweighted, r.weighted,
                     "yes" if abs(r.weighted) >= C.SMD_THRESHOLD else "no")
                    for r in covariates],
                   columns=["covariate", "role", "SMD unweighted", "SMD weighted", "exceeds"])
smd.sort_values("SMD weighted", key=abs, ascending=False).head(8)

## 3. Panel B — where the arms overlap

Pooled overlap can look acceptable while treatment is nearly determined by centre, so `[§9]` reports
it **within each centre as well**. A centre with structural non-positivity is reported as such and
given no overlap plot.

In [ ]:
reported = [c for c in bundle.balance.centres if c.status == bundle.balance.pooled.status]
omitted = [c for c in bundle.balance.centres if c.status != bundle.balance.pooled.status]

pd.DataFrame([(c.centre, c.in_cohort, c.weighted, c.n[0], c.n[1], c.ess[0], c.ess[1], c.status)
              for c in (*reported, *omitted)],
             columns=["centre", "in cohort", "weighted", f"n {ARM[0]}", f"n {ARM[1]}",
                      f"ESS {ARM[0]}", f"ESS {ARM[1]}", "status"])

## 4. The figure

Panel A on the left, one overlap histogram per reported centre on the right.

In [ ]:
fig = plt.figure(figsize=(7.2, 4.4))
grid = fig.add_gridspec(len(reported), 2, width_ratios=[1.35, 1], wspace=0.5, hspace=0.75)

ax_love = fig.add_subplot(grid[:, 0])
handles, labels = M.love(ax_love, covariates)
# The weighted entry is two markers: blue within [§9]'s threshold, red at or above it.
ax_love.legend(handles, labels, handler_map=M.PAIRED_HANDLER, loc="lower right", frameon=False)

panels = []
for i, centre in enumerate(reported):
    ax = fig.add_subplot(grid[i, 1])
    M.overlap(ax, bundle, centre.centre)          # headed by the centre, and nothing else
    panels.append(ax)
    if i == 0:
        ax.legend(frameon=False)
    if i < len(reported) - 1:
        ax.set_xlabel("")

# The letters name the panels; the panels' own headings say what each one is.
M.panel_letters(fig, {"A": ax_love, "B": panels[0]})

## 5. Write it

In [ ]:
support = "; ".join(f"{c.centre} {c.n[1]} {ARM[1]} vs {c.n[0]} {ARM[0]}" for c in reported)
caption = (
    "Figure 2. Design diagnostics. (A) Absolute standardised mean difference for every [§6] "
    "covariate in a propensity model or held out as a negative control, before and after overlap "
    "weighting; the dotted line is [§9]'s 0.10 threshold and red marks a covariate still above it. "
    "Penumbra volume, excluded from every model as ischaemic core and Tmax>6 s restated, is in the "
    "Supplement. (B) Propensity-score distribution by arm within each "
    f"centre that contributes a weighted contrast, the weighted cohort being {support}.")

panel_a = [("A", M.label(r.covariate), quantity, value)
           for r in covariates
           for quantity, value in (("SMD unweighted", r.unweighted), ("SMD weighted", r.weighted))]
panel_b = [("B", c.centre, quantity, value)
           for c in reported
           for quantity, value in ((f"n {ARM[0]}", c.n[0]), (f"n {ARM[1]}", c.n[1]),
                                   (f"ESS {ARM[0]}", c.ess[0]), (f"ESS {ARM[1]}", c.ess[1]),
                                   ("e min", min(c.e_range[a][0] for a in ARMS)),
                                   ("e max", max(c.e_range[a][1] for a in ARMS)))]
source = pd.DataFrame(panel_a + panel_b, columns=["panel", "row", "quantity", "value"])

path = M.save_figure("Figure_2", fig, source, (
    caption,
    report.exceedance_clause(bundle.balance),
    "Centres with structural non-positivity or no weighted contrast are given no overlap plot [§9]: "
    + ("; ".join(f"{c.centre} ({c.status})" for c in omitted) if omitted else "none") + ".",
    report.ATO_DESCRIPTION,
    M.provenance_clause(bundle)))

M.save_legends()      # rebuilds out/manuscript/figure_legends.md
M.show(path)

## 6. Reading it

- **The exceedances are the point of the panel.** Overlap weights balance exactly only in an
  unpenalised fit; near-separation forces `[§7]`'s Firth penalty, and the penalty is why the exact
  property does not hold here `[§9 amendment]`. The covariates still above the threshold are named in
  the caption, and `[§13]`'s full-covariate arm — Table S1 — is what sizes them.
- **Centre is a covariate and a design fact at once.** It carries this cohort's worst residual
  imbalance, and panel B is why: the arms are not merely unbalanced within centres, they are
  differently *supported* across them.
- **A centre with no bridging patient is not a small-sample problem.** It is a population where the
  contrast does not exist, which is why it is excluded rather than down-weighted `[§3]`.